# Dunker Hallbar Metal-Insulator-Semiconductor Measurement

In [1]:
import time
import numpy as np
import pandas as pd
import pyvisa as visa
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt
from qcodes import (
    Parameter,
    Measurement,
    Station,
    load_or_create_experiment,
    initialise_or_create_database_at,
)
from qstl_instruments.qstl_qdac2 import QSTL_QDac2
from qstl_instruments.qstl_nidaq import QSTL_NIDaq

from pymeasure.instruments.keithley import Keithley2400
%matplotlib inline

rm = visa.ResourceManager()
pprint(rm.list_resources())

visa_addr_keithley = "GPIB0::3::INSTR"

# led = Keithley2400(visa_addr_keithley)

('ASRL1::INSTR',
 'ASRL3::INSTR',
 'ASRL5::INSTR',
 'ASRL11::INSTR',
 'ASRL20::INSTR',
 'ASRL21::INSTR',
 'ASRL22::INSTR',
 'ASRL23::INSTR',
 'ASRL24::INSTR',
 'ASRL25::INSTR',
 'ASRL26::INSTR',
 'ASRL27::INSTR')


In [2]:
contacts = {
    "gate" : 1,
    "bias" : 2,
}

qdac2 = QSTL_QDac2(
    name = "qdac2",
    address = "ASRL5::INSTR",
    ramp_rate = 1,
    i_threshold = 2e-9,
    v_limit = 5.0,
    contacts = contacts
)

daq = QSTL_NIDaq(
    max_sampling_rate = int(1e6),
    gain = 1e8
)

station = Station(qdac2)
qdac2.ramp_all_channels_to_zero()
qdac2.get_initial_voltages()

Connected to: QDevil QDAC-II (serial:368, firmware:13-1.57) in 0.06s


{'gate': 0.0, 'bias': 0.0}

In [ ]:
reset_led_case = True
full_swing_meas = True
mis_measurement = True
reset_voltage = 0
bias = 1e-3
start_voltage = 2.0
end_voltage = -5.0
step_num = 50
step_time = 1e-2
vmin_num = 30

initialise_or_create_database_at("C:/Users/Measurement6/Nextcloud2/Lab/Data/QSim/2025/20251118_led_reset/hallbar_mis.db")

if reset_led_case:
    if full_swing_meas:
        exp = load_or_create_experiment("2D sweep", f"led_reset_voltage_{reset_voltage}V_full_swing")
    else:
        exp = load_or_create_experiment("2D sweep", f"led_reset_voltage_{reset_voltage}V")
    qdac2.ramp_channels(["bias"], [bias])
    qdac2.ramp_channels(["gate"], [reset_voltage])
else:
    if full_swing_meas:
        exp = load_or_create_experiment("2D sweep", f"led_reset_none_full_swing")
    else:
        exp = load_or_create_experiment("2D sweep", f"led_reset_none")

if mis_measurement:
    qdac2.ramp_channels(["bias"], [0])
    if reset_led_case:
        exp = load_or_create_experiment("2D sweep", f"MIS_meas_reset_{reset_voltage}V")
    else:
        exp = load_or_create_experiment("2D sweep", f"MIS_meas")

meas = Measurement(exp=exp, station=station)

gate_voltage = Parameter(name = "gate_voltage", label = "Gate Voltage", unit = "V")
vmin_voltage = Parameter(name = "vmin_voltage", label = "Minimum Voltage", unit = "V")
ds_current = Parameter(name = "ds_current", label = "Drain-Source Current", unit = "A")
meas.register_parameter(gate_voltage)
meas.register_parameter(vmin_voltage)
meas.register_parameter(ds_current, setpoints=(gate_voltage, vmin_voltage))

## Vmin Decreasing Measurement

In [35]:
import json
qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")
qdac2.free_all_triggers()
qdac2.ramp_all_channels_to_zero()


vmin_list = np.linspace(start_voltage, end_voltage, vmin_num + 2)
vmin_list = vmin_list[1:-1]

arrangement = qdac2.arrange(
    contacts= {
        "gate": 1
    },
    output_triggers={
        "NIDAQ" : 5,
    }
)

InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "bias" : bias,
                "reset_time" : 30,
                "reset_current" : 10e-3,
                "step_time" : step_time
            }
        )
    )
    for vmin in vmin_list:
        qdac2.free_all_triggers()
        sweep = arrangement.virtual_detune(
            contacts = tuple(["gate"]),
            start_V = tuple([start_voltage]),
            end_V = tuple([vmin]),
            steps = step_num,
            step_trigger = "NIDAQ",
            step_time_s = step_time,
            repetitions = 1
        )
        qdac2.ramp_channels(["bias"], [bias])
        # Read NI Daq traces
        result = daq.read_triggered_voltage(
            sweep,
            "Dev2/ai1",
            int(step_time * step_num * daq.max_sampling_rate),
            -10,
            +10,
            10
        )
        result = daq.reshape_array(result, step_num)
        result = np.array(result)
        result = daq.convert_volts_to_amps(result)

        datasaver.add_result(
            (vmin_voltage, [vmin]*step_num),
            (gate_voltage, np.linspace(start_voltage, vmin, step_num)),
            (ds_current, result)
        )

        loop_counter = loop_counter+1
        print(f'Time elapsed: {np.round(time.time()-start_time, 2)} sec. Loop finished: {loop_counter}/{len(vmin_list)}.')

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 43. 
Time elapsed: 0.58 sec. Loop finished: 1/30.
Time elapsed: 1.14 sec. Loop finished: 2/30.
Time elapsed: 1.69 sec. Loop finished: 3/30.
Time elapsed: 2.25 sec. Loop finished: 4/30.
Time elapsed: 2.79 sec. Loop finished: 5/30.
Time elapsed: 3.34 sec. Loop finished: 6/30.
Time elapsed: 3.89 sec. Loop finished: 7/30.
Time elapsed: 4.44 sec. Loop finished: 8/30.
Time elapsed: 4.99 sec. Loop finished: 9/30.
Time elapsed: 5.55 sec. Loop finished: 10/30.
Time elapsed: 6.1 sec. Loop finished: 11/30.
Time elapsed: 6.65 sec. Loop finished: 12/30.
Time elapsed: 7.19 sec. Loop finished: 13/30.
Time elapsed: 7.74 sec. Loop finished: 14/30.
Time elapsed: 8.28 sec. Loop finished: 15/30.
Time elapsed: 8.83 sec. Loop finished: 16/30.
Time elapsed: 9.37 sec. Loop finished: 17/30.
Time elapsed: 9.93 sec. Loop finished: 18/30.
Time elapsed: 10.49 sec. Loop finished: 19/30.
Time elapsed: 11.06 sec. Loop finished: 20/30.
Time elapsed: 11.62 sec. Loop finished: 21/30.
T

## Full Swing Measurement

In [16]:
import json
qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")
qdac2.free_all_triggers()
qdac2.ramp_all_channels_to_zero()

step_time = 0.1

arrangement = qdac2.arrange(
    contacts= {
        "gate": 1
    },
    output_triggers={
        "NIDAQ" : 5,
    }
)

InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "bias" : bias,
                "reset_time" : 30,
                "reset_current" : 10e-3,
                "step_time" : step_time
            }
        )
    )
    qdac2.free_all_triggers()
    sweep = arrangement.virtual_detune(
        contacts = tuple(["gate"]),
        start_V = tuple([start_voltage]),
        end_V = tuple([end_voltage]),
        steps = step_num,
        step_trigger = "NIDAQ",
        step_time_s = step_time,
        repetitions = 1
    )
    qdac2.ramp_channels(["bias"], [bias])
    # Read NI Daq traces
    result = daq.read_triggered_voltage(
        sweep,
        "Dev2/ai1",
        int(step_time * step_num * daq.max_sampling_rate),
        -10,
        +10,
        100
    )
    result = daq.reshape_array(result, step_num)
    result = np.array(result)
    result = daq.convert_volts_to_amps(result)

    datasaver.add_result(
        (vmin_voltage, [0]*step_num),
        (gate_voltage, np.linspace(start_voltage, end_voltage, step_num)),
        (ds_current, result)
    )

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 4. 
Time elapsed: 5.39 sec.


## MIS Current Measure (Bias 0)

In [13]:
import json
qdac2.channels[0:23].dc_slew_rate_V_per_s("inf")
qdac2.free_all_triggers()
qdac2.ramp_all_channels_to_zero()

step_time = 0.1

arrangement = qdac2.arrange(
    contacts= {
        "gate": 1
    },
    output_triggers={
        "NIDAQ" : 5,
    }
)

InitialConditions = qdac2.get_initial_voltages()

start_time = time.time()
with meas.run() as datasaver:
    loop_counter = 0
    datasaver.dataset.add_metadata(tag="Contacts", metadata=json.dumps(contacts))
    datasaver.dataset.add_metadata(tag="IC", metadata=json.dumps(InitialConditions))
    datasaver.dataset.add_metadata(
        tag = "Sweep_params",
        metadata = json.dumps(
            {
                "bias" : 0,
                "reset_time" : 30,
                "reset_current" : 10e-3,
                "step_time" : step_time
            }
        )
    )
    qdac2.free_all_triggers()
    sweep = arrangement.virtual_detune(
        contacts = tuple(["gate"]),
        start_V = tuple([start_voltage]),
        end_V = tuple([end_voltage]),
        steps = step_num,
        step_trigger = "NIDAQ",
        step_time_s = step_time,
        repetitions = 1
    )
    qdac2.ramp_channels(["bias"], [0])
    # Read NI Daq traces
    result = daq.read_triggered_voltage(
        sweep,
        "Dev2/ai1",
        int(step_time * step_num * daq.max_sampling_rate),
        -10,
        +10,
        100
    )
    result = daq.reshape_array(result, step_num)
    result = np.array(result)
    result = daq.convert_volts_to_amps(result)

    datasaver.add_result(
        (vmin_voltage, [0]*step_num),
        (gate_voltage, np.linspace(start_voltage, end_voltage, step_num)),
        (ds_current, result)
    )

end_time = time.time()
print(f'Time elapsed: {np.round(end_time-start_time, 2)} sec.')

qdac2.channels[0:23].dc_slew_rate_V_per_s(1)

qdac2.ramp_all_channels_to_zero()

Starting experimental run with id: 3. 
Time elapsed: 5.31 sec.


In [15]:
qdac2.ramp_channels(["bias"], [bias])

0.001